# Final test and eval

In [ ]:
# packages and libs
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone

In [2]:
# Leading feature datasets
project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

features_path = (
    project_root
    / 'data'
    / 'processed'
    / 'ca1_model_features.parquet'
)

models_directory = (
    project_root
    / 'models'
)

reports_directory = (
    project_root
    / 'reports'
)

feature_df = pd.read_parquet(
    features_path
)

feature_df['date'] = pd.to_datetime(
    feature_df['date']
)

feature_df = feature_df.sort_values(
    ['store_id', 'item_id', 'date']
).reset_index(drop=True)

print(f'Rows: {len(feature_df):,}')
print(f'Items: {feature_df["item_id"].nunique():,}')

Rows: 166,886
Items: 108


### Setting the final development and testing periods

In [3]:
all_dates = pd.DatetimeIndex(
    sorted(feature_df['date'].unique())
)

development_dates = all_dates[:-28]
test_dates = all_dates[-28:]

development_df = feature_df.loc[
    feature_df['date'].isin(
        development_dates
    )
].copy()

test_df = feature_df.loc[
    feature_df['date'].isin(
        test_dates
    )
].copy()

print(
    f'Development: '
    f'{development_dates.min().date()} '
    f'to {development_dates.max().date()}'
)

print(
    f'Final test: '
    f'{test_dates.min().date()} '
    f'to {test_dates.max().date()}'
)

print(
    f'Development rows: '
    f'{len(development_df):,}'
)

print(
    f'Test rows: '
    f'{len(test_df):,}'
)

Development: 2011-02-26 to 2016-04-24
Final test: 2016-04-25 to 2016-05-22
Development rows: 163,862
Test rows: 3,024


Training and validation now become the complete development dataset. The final 28 days remain the test period.

In [4]:
# Use the same predictors as the previous process
numeric_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7',
    'sell_price',
    'price_change',
    'event_indicator'
]

categorical_features = [
    'weekday',
    'month',
    'item_id',
    'dept_id',
    'cat_id'
]

model_features = (
    numeric_features
    + categorical_features
)

target = 'sales'

X_development = development_df[
    model_features
]

y_development = development_df[
    target
]

series_keys = [
    'store_id',
    'item_id'
]

Using the same functions and prediction metrics and previous steps

In [5]:
def recursive_ml_forecast(
    model,
    data,
    forecast_dates,
    feature_columns,
    model_name
):
    forecast_dates = pd.DatetimeIndex(
        forecast_dates
    )

    forecast_start = forecast_dates.min()

    histories = {}

    for series_values, group in data.groupby(
        series_keys,
        sort=False
    ):
        history = (
            group.loc[
                group['date'] < forecast_start,
                'sales'
            ]
            .astype(float)
            .tolist()
        )

        if len(history) < 28:
            raise ValueError(
                f'{series_values} has fewer than '
                f'28 historical observations.'
            )

        histories[series_values] = history

    prediction_frames = []

    for forecast_date in forecast_dates:
        day_rows = data.loc[
            data['date'] == forecast_date
        ].sort_values(
            series_keys
        ).copy()

        for row_index, row in day_rows.iterrows():
            series_values = tuple(
                row[column]
                for column in series_keys
            )

            history = histories[
                series_values
            ]

            day_rows.at[
                row_index,
                'lag_1'
            ] = history[-1]

            day_rows.at[
                row_index,
                'lag_7'
            ] = history[-7]

            day_rows.at[
                row_index,
                'lag_14'
            ] = history[-14]

            day_rows.at[
                row_index,
                'lag_28'
            ] = history[-28]

            day_rows.at[
                row_index,
                'rolling_mean_7'
            ] = np.mean(history[-7:])

            day_rows.at[
                row_index,
                'rolling_mean_28'
            ] = np.mean(history[-28:])

            day_rows.at[
                row_index,
                'rolling_std_7'
            ] = np.std(
                history[-7:],
                ddof=1
            )

        assert not day_rows[
            feature_columns
        ].isna().any().any()

        raw_forecasts = model.predict(
            day_rows[feature_columns]
        )

        # Sales cannot be negative.
        clipped_forecasts = np.maximum(
            raw_forecasts,
            0
        )

        output = day_rows[
            [
                'store_id',
                'item_id',
                'dept_id',
                'cat_id',
                'date',
                'sales'
            ]
        ].copy()

        output['model'] = model_name
        output['raw_forecast'] = raw_forecasts
        output['forecast'] = clipped_forecasts

        prediction_frames.append(output)

        # Use clipped predictions to construct
        # subsequent forecast-day features.
        for row, prediction in zip(
            day_rows.itertuples(index=False),
            clipped_forecasts
        ):
            series_values = (
                row.store_id,
                row.item_id
            )

            histories[
                series_values
            ].append(float(prediction))

    return pd.concat(
        prediction_frames,
        ignore_index=True
    )

In [6]:
def calculate_prediction_metrics(
    predictions,
    prediction_column
):
    actual = predictions[
        'sales'
    ].to_numpy(dtype=float)

    forecast = predictions[
        prediction_column
    ].to_numpy(dtype=float)

    errors = actual - forecast

    mae = np.mean(
        np.abs(errors)
    )

    rmse = np.sqrt(
        np.mean(errors ** 2)
    )

    denominator = np.sum(
        np.abs(actual)
    )

    wape = (
        np.sum(np.abs(errors))
        / denominator
        if denominator > 0
        else np.nan
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape
    }

In [7]:
# Baseline forecasting function
def recursive_baseline_forecast(
    data,
    forecast_dates,
    method,
    model_name
):
    forecast_dates = pd.DatetimeIndex(
        forecast_dates
    )

    forecast_start = forecast_dates.min()

    histories = {}

    for series_values, group in data.groupby(
        series_keys,
        sort=False
    ):
        histories[series_values] = (
            group.loc[
                group['date'] < forecast_start,
                'sales'
            ]
            .astype(float)
            .tolist()
        )

    prediction_frames = []

    for forecast_date in forecast_dates:
        day_rows = data.loc[
            data['date'] == forecast_date
        ].sort_values(
            series_keys
        ).copy()

        forecasts = []

        for row in day_rows.itertuples(
            index=False
        ):
            series_values = (
                row.store_id,
                row.item_id
            )

            history = histories[
                series_values
            ]

            if method == 'previous_day':
                prediction = history[-1]

            elif method == 'seasonal_naive_7':
                prediction = history[-7]

            elif method == 'rolling_mean_28':
                prediction = np.mean(
                    history[-28:]
                )

            else:
                raise ValueError(
                    f'Unknown method: {method}'
                )

            prediction = max(
                float(prediction),
                0
            )

            forecasts.append(prediction)

            histories[
                series_values
            ].append(prediction)

        output = day_rows[
            [
                'store_id',
                'item_id',
                'dept_id',
                'cat_id',
                'date',
                'sales'
            ]
        ].copy()

        output['model'] = model_name
        output['raw_forecast'] = forecasts
        output['forecast'] = forecasts

        prediction_frames.append(output)

    return pd.concat(
        prediction_frames,
        ignore_index=True
    )

In [8]:
# Final baseline forecast
test_prediction_frames = []

baseline_methods = [
    'previous_day',
    'seasonal_naive_7',
    'rolling_mean_28'
]

for method in baseline_methods:
    print(f'Forecasting {method}...')

    predictions = (
        recursive_baseline_forecast(
            data=feature_df,
            forecast_dates=test_dates,
            method=method,
            model_name=method
        )
    )

    test_prediction_frames.append(
        predictions
    )

Forecasting previous_day...
Forecasting seasonal_naive_7...
Forecasting rolling_mean_28...


In [9]:
# Refit the fixed models
model_files = {
    'linear_regression':
        'linear_regression_pipeline.joblib',
    'ridge_regression':
        'ridge_regression_pipeline.joblib',
    'random_forest_tuned':
        'random_forest_tuned_pipeline.joblib',
    'xgboost_tuned':
        'xgboost_tuned_pipeline.joblib'
}

for model_name, filename in (
    model_files.items()
):
    model_path = (
        models_directory
        / filename
    )

    print(
        f'{model_name}: '
        f'{model_path.exists()}'
    )

linear_regression: True
ridge_regression: True
random_forest_tuned: True
xgboost_tuned: True


In [10]:
# Retrain using development data
final_models = {}

for model_name, filename in (
    model_files.items()
):
    print(f'Training {model_name}...')

    model_template = joblib.load(
        models_directory
        / filename
    )

    final_model = clone(
        model_template
    )

    final_model.fit(
        X_development,
        y_development
    )

    predictions = recursive_ml_forecast(
        model=final_model,
        data=feature_df,
        forecast_dates=test_dates,
        feature_columns=model_features,
        model_name=model_name
    )

    final_models[
        model_name
    ] = final_model

    test_prediction_frames.append(
        predictions
    )

    print(f'Finished {model_name}.')

Training linear_regression...
Finished linear_regression.
Training ridge_regression...
Finished ridge_regression.
Training random_forest_tuned...
Finished random_forest_tuned.
Training xgboost_tuned...
Finished xgboost_tuned.


In [11]:
# Combining test predictions
test_predictions = pd.concat(
    test_prediction_frames,
    ignore_index=True
)

print(
    f'Total prediction rows: '
    f'{len(test_predictions):,}'
)

print(
    test_predictions[
        'model'
    ].value_counts()
)

Total prediction rows: 21,168
model
previous_day           3024
seasonal_naive_7       3024
rolling_mean_28        3024
linear_regression      3024
ridge_regression       3024
random_forest_tuned    3024
xgboost_tuned          3024
Name: count, dtype: int64


In [12]:
# Calculating final test metrics for each model
test_metric_rows = []

for model_name, predictions in (
    test_predictions.groupby('model')
):
    metrics = calculate_prediction_metrics(
        predictions,
        'forecast'
    )

    negative_predictions = (
        predictions[
            'raw_forecast'
        ].lt(0).sum()
    )

    test_metric_rows.append({
        'model': model_name,
        **metrics,
        'negative_raw_predictions':
            negative_predictions
    })

test_metrics = (
    pd.DataFrame(test_metric_rows)
    .sort_values(
        ['WAPE', 'RMSE']
    )
    .reset_index(drop=True)
)

test_metrics[
    'WAPE_percentage'
] = (
    test_metrics['WAPE']
    * 100
)

test_metrics

,model,MAE,RMSE,WAPE,negative_raw_predictions,WAPE_percentage
0,random_forest_tuned,1.273318,2.666395,0.764750,0,76.474972
1,rolling_mean_28,1.275715,2.712615,0.766189,0,76.618906
2,xgboost_tuned,1.312323,2.677344,0.788176,0,78.817553
3,linear_regression,1.356158,2.734692,0.814503,156,81.450270
4,ridge_regression,1.360844,2.734023,0.817317,148,81.731737
5,seasonal_naive_7,1.449405,3.314580,0.870506,0,87.050645
6,previous_day,1.596892,3.242258,0.959086,0,95.908640


In [13]:
# Combining validation and test performance
validation_metrics = pd.read_csv(
    reports_directory
    / 'validation_model_comparison_tuned.csv'
)

validation_test_comparison = (
    test_metrics.merge(
        validation_metrics[
            [
                'model',
                'MAE',
                'RMSE',
                'WAPE'
            ]
        ],
        on='model',
        how='left',
        suffixes=(
            '_test',
            '_validation'
        )
    )
)

validation_test_comparison[
    'WAPE_change'
] = (
    validation_test_comparison[
        'WAPE_test'
    ]
    - validation_test_comparison[
        'WAPE_validation'
    ]
)

validation_test_comparison

,model,MAE_test,RMSE_test,WAPE_test,negative_raw_predictions,WAPE_percentage,MAE_validation,RMSE_validation,WAPE_validation,WAPE_change
0,random_forest_tuned,1.273318,2.666395,0.764750,0,76.474972,1.177787,2.278336,0.751715,0.013034
1,rolling_mean_28,1.275715,2.712615,0.766189,0,76.618906,1.169400,2.417961,0.746363,0.019827
2,xgboost_tuned,1.312323,2.677344,0.788176,0,78.817553,1.247358,2.408888,0.796119,-0.007943
3,linear_regression,1.356158,2.734692,0.814503,156,81.450270,1.323802,2.459765,0.844908,-0.030406
4,ridge_regression,1.360844,2.734023,0.817317,148,81.731737,1.328547,2.462850,0.847937,-0.030620
5,seasonal_naive_7,1.449405,3.314580,0.870506,0,87.050645,1.432540,3.151173,0.914310,-0.043803
6,previous_day,1.596892,3.242258,0.959086,0,95.908640,1.507937,3.461905,0.962431,-0.003345


In [14]:
# saving progress and results
reports_directory.mkdir(
    parents=True,
    exist_ok=True
)

test_metrics.to_csv(
    reports_directory
    / 'final_test_metrics.csv',
    index=False
)

validation_test_comparison.to_csv(
    reports_directory
    / 'validation_test_comparison.csv',
    index=False
)

test_predictions.to_parquet(
    project_root
    / 'data'
    / 'processed'
    / 'ca1_final_test_predictions.parquet',
    index=False
)

for model_name, model in (
    final_models.items()
):
    joblib.dump(
        model,
        models_directory
        / f'final_{model_name}_pipeline.joblib',
        compress=3
    )

print('Final test evaluation saved.')

Final test evaluation saved.
